# Projet final : les résultats d'examens

**Équipe :** (nom de l'équipe et des 5 membres)

Un réseau de 25 écoles (fictives) vous confie les résultats de ses candidats aux examens de fin d'études secondaires, en 2024 et 2025. Le fichier sort tel quel du secrétariat. Le directeur veut savoir quelles filières, quels secteurs et quelles communes réussissent, et lesquelles décrochent. Un candidat réussit avec une moyenne d'au moins 10 sur 20.

Ce notebook montre **votre démarche** : chaque étape dit ce que vous cherchez, le code, le résultat et ce que vous en concluez. Les graphiques doivent être visibles : avant de pousser, **Restart**, puis **Run All**.

Le tableau de bord (`app.py`) reprend ensuite vos résultats pour le décideur.

In [4]:
import matplotlib.pyplot as plt
import pandas as pd

brut = pd.read_csv("donnees/resultats_brut.csv")

## 1. Diagnostic

Forme, types, valeurs manquantes, lignes en double, et toutes les écritures d'une même valeur dans les colonnes de texte. Notez chaque problème trouvé : vous le corrigerez à l'étape 2.

In [16]:
print("Forme :", brut.shape)
print("\nTypes des colonnes :")
print(brut.dtypes)
print("\nValeurs manquantes par colonne :")
print(brut.isna().sum())
print("\nLignes en double :", brut.duplicated().sum())

print("\nÉcritures différentes dans les colonnes de texte :")
for colonne in ["code_ecole", "filiere", "sexe"]:
    print(f"- {colonne} : {brut[colonne].nunique()} écritures")

print("Filières :", sorted(brut["filiere"].unique()))
print("Sexes :", sorted(brut["sexe"].unique()))

# Moyennes qui ne sont pas des nombres (ex. "11,6" ou "11.2/20")
converti = pd.to_numeric(brut["moyenne"], errors="coerce")
non_convertibles = brut.loc[converti.isna() & brut["moyenne"].notna(), "moyenne"]
print("\nMoyennes non numériques :", non_convertibles.nunique())
print(non_convertibles.drop_duplicates().head(15).tolist())

Forme : (1529, 6)

Types des colonnes :
candidat        str
annee         int64
code_ecole      str
filiere         str
sexe            str
moyenne         str
dtype: object

Valeurs manquantes par colonne :
candidat       0
annee          0
code_ecole     0
filiere        0
sexe           0
moyenne       18
dtype: int64

Lignes en double : 25

Écritures différentes dans les colonnes de texte :
- code_ecole : 67 écritures
- filiere : 16 écritures
- sexe : 2 écritures
Filières : [' LLA', ' SES', ' SMP', ' SVT', 'LLA', 'LLA ', 'SES', 'SES ', 'SMP', 'SMP ', 'SVT', 'SVT ', 'lla', 'ses', 'smp', 'svt']
Sexes : ['F', 'M']

Moyennes non numériques : 108
['11,6', '11.2/20', '8,8', '10,2', '13.7/20', '12,4', '10.3/20', '14.0/20', '11.4/20', '8,5', '8,1', '13,3', '7,4', '10,7', '7.0/20']


## 2. Nettoyage

Une copie `propre`, puis une cellule par problème, en comptant les lignes avant et après chaque suppression. Dites ce que vous supprimez et pourquoi.

**Contrôle :** à la fin, `propre.shape` vaut `(1486, 6)`, sans valeur manquante ni doublon, avec 4 valeurs dans `filiere`, 2 valeurs dans `sexe`.

In [17]:
propre = brut.copy()

# a) Textes : enlever les espaces, tout mettre en majuscules
for colonne in ["code_ecole", "filiere", "sexe"]:
    propre[colonne] = propre[colonne].astype(str).str.strip().str.upper()

# b) Moyenne : retirer "/20", virgule -> point, convertir en nombre
propre["moyenne"] = (
    propre["moyenne"].astype(str)
    .str.replace("/20", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
propre["moyenne"] = pd.to_numeric(propre["moyenne"], errors="coerce")

# c) Doublons
avant = len(propre)
propre = propre.drop_duplicates()
print(f"Doublons supprimés : {avant - len(propre)}")          # 25

# d) Lignes sans moyenne
avant = len(propre)
propre = propre.dropna(subset=["moyenne"]).reset_index(drop=True)
print(f"Lignes sans moyenne supprimées : {avant - len(propre)}")  # 18

# Contrôle : (1486, 6), 0 manquant, 0 doublon, 4 filières, 2 sexes
print(propre.shape, int(propre.isna().sum().sum()), int(propre.duplicated().sum()))
print(propre["filiere"].nunique(), propre["sexe"].nunique())

Doublons supprimés : 25
Lignes sans moyenne supprimées : 18
(1486, 6) 0 0
4 2


## 3. Jointures

Joignez `propre` avec `ecoles.csv`, `communes.csv`, et ajoutez les colonnes calculées dont vous aurez besoin. Comptez les lignes avant et après chaque `merge`.

**Contrôle :** 1486 lignes après les jointures, autant qu'avant.

In [18]:
ecoles = pd.read_csv(DOSSIER / "ecoles.csv")
communes = pd.read_csv(DOSSIER / "communes.csv")

print("Avant jointures :", len(propre))

# how="left" : on garde tous les candidats, même sans école connue
complet = propre.merge(ecoles, on="code_ecole", how="left")
print("Après jointure ecoles :", len(complet))
print("Codes sans école :", complet["ecole"].isna().sum())      # 27

complet = complet.merge(communes, on="commune", how="left")
print("Après jointure communes :", len(complet))                # 1486

# Colonne calculée : réussite si moyenne >= 10
complet["reussite"] = (complet["moyenne"] >= 10).astype(int)

# Le code absent (E25) : on garde les candidats, marqués "Inconnu"
complet[["commune", "secteur", "departement"]] = complet[
    ["commune", "secteur", "departement"]
].fillna("Inconnu")
complet["ecole"] = complet["ecole"].fillna("E25 (absente de ecoles.csv)")

# Contrôle : autant de lignes qu'avant les jointures
assert len(complet) == len(propre) == 1486

# Question 5 : quel code est absent ?
print(sorted(set(complet["code_ecole"]) - set(ecoles["code_ecole"])))  # ['E25']

# Enregistrement du fichier lu par app.py
complet.to_csv(DOSSIER / "resultats_propre.csv", index=False)

Avant jointures : 1486
Après jointure ecoles : 1486
Codes sans école : 27
Après jointure communes : 1486
['E25']


Enregistrez le tableau complet : c'est le fichier que lit le tableau de bord.

```python
complet.to_csv("donnees/resultats_propre.csv", index=False)
```

Codes absents de ecoles.csv : ['E25']
E25 : 25 candidats


## 4. Quelles filières réussissent le mieux ? L'écart entre filières est-il le même en 2024 et en 2025 ?

**Ce qu'on en conclut :**

## 5. Le secteur de l'école (public, privé, congréganiste) fait-il une différence ? Montrez-le avec un graphique.

**Ce qu'on en conclut :**

## 6. Quelles communes et quels départements décrochent ? Lesquels progressent ?

**Ce qu'on en conclut :**

## 7. Filles et garçons réussissent-ils de la même façon, dans chaque filière ?

**Ce qu'on en conclut :**

## 8. Un code d'école des résultats est absent de `ecoles.csv` : lequel, combien de candidats, et comment l'avez-vous traité ?

**Ce qu'on en conclut :**

## 9. Conclusion

Cinq phrases au plus : ce qu'il faut retenir, ce que les données ne permettent pas de dire, ce qu'il faudrait mesurer ensuite.